# `11 — Dijkstra’s algorithm (non-negative weights)`

## **When to use**
- Weighted graph with **non-negative** edge weights.
- Find shortest distances from one source **s** to all vertices.

## **Complexity summary**
Let V = number of vertices, E = number of edges.

| Implementation | Time |
|---|---:|
| naive "pick min d in Q" | O(V^2 + E) |
| using heap events (priority queue) | **O(E log V)** |

## **Key invariant (core of proof)**
We maintain:
- **S** = vertices whose shortest distance is already finalized
- **Q = V \\ S** = remaining vertices

Invariant:
1) For every v in **S**: **d[v] = ρ(s, v)** (final and correct)
2) For every v in **Q**: $d[v] = \min_{u \in S} \left( \rho(s,u) + w(u,v) \right)$
   (best known path to v going from S with one crossing edge)

Algorithm step:
- choose v* in Q with minimal d[v*]
- add v* to S
- relax all outgoing edges of v* to update d for Q

## **Why v* is correct (proof sketch)**
Assume there exists a cheaper path p to v* than d[v*].
That path must leave S at some first vertex u in Q.
Since weights are non-negative, the remaining path from u to v* adds ≥ 0.

So $w(p) ≥ d[u]$.

But we picked v* with minimal d in Q, so d[v*] ≤ d[u].
Thus w(p) ≥ d[v*], contradiction.
Therefore d[v*] = ρ(s, v*).

In [3]:
import heapq
from typing import Optional, Dict, List, Tuple


Graph = Dict[int, Dict[int, float]]


def show_weighted_graph(G: Graph) -> None:
    print("-" * 70)
    print("Graph (adjacency dict):")
    print("-" * 70)
    for v in sorted(G.keys()):
        outs = ", ".join([f"{v}->{u} (w={G[v][u]})" for u in sorted(G[v].keys())])
        print(f"{v}: {outs}")


def dijkstra_visual(G: Graph, *, s: int) -> Tuple[List[float], List[Optional[int]]]:
    n: int = len(G)
    INF: float = float("inf")

    d: List[float] = [INF] * n
    parent: List[Optional[int]] = [None] * n
    S: set[int] = set()  # finalized

    h: List[Tuple[float, int]] = []
    d[s] = 0.0
    heapq.heappush(h, (d[s], s))

    print("-" * 80)
    print(f"Dijkstra (heap events) from source s={s}")
    print("-" * 80)
    print("Legend:")
    print("  - pop (dist, v) from heap")
    print("  - if v already finalized -> outdated event, skip")
    print("  - else finalize v, relax outgoing edges")
    print("-" * 80)

    step: int = 0
    while h:
        dist_v, v = heapq.heappop(h)

        print(f"\nStep {step}: heappop -> (dist={dist_v}, v={v})")
        step += 1

        if v in S:
            print("  outdated event (v already in S) -> skip")
            continue

        S.add(v)
        print(f"  finalize v={v}  => d[{v}] = {d[v]}")
        print(f"  S = {sorted(S)}")

        # Relax edges v -> u:
        for u in sorted(G[v].keys()):
            w = G[v][u]
            new_d = d[v] + w
            print(f"    relax edge {v}->{u} (w={w}): try {new_d} vs current d[{u}]={d[u]}")
            if new_d < d[u]:
                d[u] = new_d
                parent[u] = v
                heapq.heappush(h, (new_d, u))
                print(f"      ✅ update: d[{u}]={d[u]}, parent[{u}]={v}, push({new_d},{u})")
            else:
                print("      no update")

        print("  current d:", d)

    return d, parent


In [ ]:
# Example:
G: Graph = {
    0: {1: 10, 2: 5},
    1: {3: 1},
    2: {1: 2, 3: 9, 4: 2},
    3: {},
    4: {0: 7, 3: 4},
}

show_weighted_graph(G)

d, parent = dijkstra_visual(G, s=0)

print("\nFinal distances:", d)
print("Parents:", parent)


----------------------------------------------------------------------
Graph (adjacency dict):
----------------------------------------------------------------------
0: 0->1 (w=10), 0->2 (w=5)
1: 1->3 (w=1)
2: 2->1 (w=2), 2->3 (w=9), 2->4 (w=2)
3: 
4: 4->0 (w=7), 4->3 (w=4)
--------------------------------------------------------------------------------
Dijkstra (heap events) from source s=0
--------------------------------------------------------------------------------
Legend:
  - pop (dist, v) from heap
  - if v already finalized -> outdated event, skip
  - else finalize v, relax outgoing edges
--------------------------------------------------------------------------------

Step 0: heappop -> (dist=0.0, v=0)
  finalize v=0  => d[0] = 0.0
  S = [0]
    relax edge 0->1 (w=10): try 10.0 vs current d[1]=inf
      ✅ update: d[1]=10.0, parent[1]=0, push(10.0,1)
    relax edge 0->2 (w=5): try 5.0 vs current d[2]=inf
      ✅ update: d[2]=5.0, parent[2]=0, push(5.0,2)
  current d: [0.0, 10.

In [5]:
from typing import List, Optional

def restore_path(parent: List[Optional[int]], *, s: int, t: int) -> List[int]:
    if s == t:
        return [s]
    if parent[t] is None:
        return []
    path: List[int] = []
    cur: Optional[int] = t
    while cur is not None:
        path.append(cur)
        if cur == s:
            break
        cur = parent[cur]
    path.reverse()
    return path

target = 3
path = restore_path(parent, s=0, t=target)
print(f"Shortest path 0 -> {target}:", path)

Shortest path 0 -> 3: [0, 2, 1, 3]
